In [1]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import KFold

import nibabel as nib
import pydicom as pdm
import nilearn as nl
import nilearn.plotting as nlplt
import h5py

import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.animation as anim
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

import seaborn as sns
import imageio
from skimage.transform import resize
from skimage.util import montage

# from IPython.display import Image as show_gif
# from IPython.display import clear_output
# from IPython.display import YouTubeVideo

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn import MSELoss

# !pip install opencv-python==4.6.0.66
# !pip install -U albumentations --no-binary qudida,albumentations
import albumentations as A
# from albumentations.pytorch import ToTensor, ToTensorV2


from albumentations import Compose, HorizontalFlip
# from albumentations.pytorch import ToTensor, ToTensorV2 

import warnings
warnings.simplefilter("ignore")

# Function to Calculate Volume of a Tumor based on mask file

In [20]:
def calculate_volume(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        outputloc = '../Results/Result/Vanilla_Unet/BraTS-GLI-'  + patient_id
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    sample_filename1 = baseloc + pefix + suffixs[0]
    sample_img1_f = nib.load(sample_filename1)
    sample_img1 = np.asarray(sample_img1_f.dataobj)
    # sample_img1 = np.rot90(sample_img1)
#     print(sample_img1.shape)

    sample_filename2 = baseloc + pefix + suffixs[1]
    sample_img2_f = nib.load(sample_filename2)
    sample_img2 = np.asarray(sample_img2_f.dataobj)
    # sample_img2  = np.rot90(sample_img2)
#     print(sample_img2.shape)

    sample_filename3 = baseloc + pefix + suffixs[2]
    sample_img3_f = nib.load(sample_filename3)
    sample_img3 = np.asarray(sample_img3_f.dataobj)
    # sample_img3  = np.rot90(sample_img3)
#     print(sample_img3.shape)

    sample_filename4 = baseloc + pefix + suffixs[3]
    sample_img4_f = nib.load(sample_filename4)
    sample_img4 = np.asarray(sample_img4_f.dataobj)
    # sample_img4  = np.rot90(sample_img4)
#     print(sample_img4.shape)

    sample_filename_mask = baseloc + pefix + suffixs[4]
    sample_mask_f = nib.load(sample_filename_mask)
    sample_mask = np.asarray(sample_mask_f.dataobj)
#     print(sample_mask.shape)
    
    output_filename_mask = outputloc + suffixs[4]
    output_mask_f = nib.load(output_filename_mask)
    output_mask = np.asarray(output_mask_f.dataobj)
#     print(output_mask.shape)

    voxel_dims = sample_mask_f.header['pixdim'][0:3]
    voxel_volume = np.prod(voxel_dims)  # typically in mm^3

    # Initialize a dictionary to hold the volumes for each tumor region
    actual_tumor_volumes_mm3 = {}
    region_label_name = {1:'NCR', 2: 'ED', 3: 'ET'}

    # Iterate through each tumor region
    for region_label in [1, 2, 3]:
        # Count the number of voxels for the current region
        region_voxel_count = np.sum(sample_mask == region_label)

        # Calculate the tumor volume for the current region and store it
        actual_tumor_volumes_mm3[region_label_name[region_label]] = region_voxel_count * voxel_volume

    # Convert the volumes to cubic centimeters (1 cm^3 = 1 mL = 1000 mm^3)
    actual_tumor_volumes_mm3 = {label: volume_mm3 / 1000 for label, volume_mm3 in actual_tumor_volumes_mm3.items()}
    
    
    # Initialize a dictionary to hold the volumes for each tumor region
    predicted_tumor_volumes_mm3 = {}
    region_label_name = {1:'NCR', 2: 'ED', 3: 'ET'}

    # Iterate through each tumor region
    for region_label in [1, 2, 3]:
        # Count the number of voxels for the current region
        region_voxel_count = np.sum(output_mask == region_label)

        # Calculate the tumor volume for the current region and store it
        predicted_tumor_volumes_mm3[region_label_name[region_label]] = region_voxel_count * voxel_volume

    # Convert the volumes to cubic centimeters (1 cm^3 = 1 mL = 1000 mm^3)
    predicted_tumor_volumes_mm3 = {label: volume_mm3 / 1000 for label, volume_mm3 in predicted_tumor_volumes_mm3.items()}

    return actual_tumor_volumes_mm3, predicted_tumor_volumes_mm3


# Calculate the Tumor Volume 

In [21]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

tumor_volume = {}

for _id in patient_ids:
    patient_id = _id.split('GLI-')[1]
    try:
        tumor_volumes, predicted_tumor_volumes = calculate_volume(dataset, patient_id)
    except FileNotFoundError:
        continue
    
    Edema = tumor_volumes['ED']
    GD_enhancing_tumor_core = tumor_volumes['ET']
    necrotic_tumor_core = tumor_volumes['NCR']
    
    WT_volume = Edema + GD_enhancing_tumor_core + necrotic_tumor_core
    TC_volume = GD_enhancing_tumor_core + necrotic_tumor_core
    ET_volume = GD_enhancing_tumor_core
    
    TC_WT_ratio = TC_volume/WT_volume
    ET_WT_ratio = ET_volume/WT_volume
    ET_TC_ratio = ET_volume/TC_volume
    
    predicted_Edema = predicted_tumor_volumes['ED']
    predicted_GD_enhancing_tumor_core = predicted_tumor_volumes['ET']
    predicted_necrotic_tumor_core = predicted_tumor_volumes['NCR']
    
    predicted_WT_volume = predicted_Edema + predicted_GD_enhancing_tumor_core + predicted_necrotic_tumor_core
    predicted_TC_volume = predicted_GD_enhancing_tumor_core + predicted_necrotic_tumor_core
    predicted_ET_volume = predicted_GD_enhancing_tumor_core
    
    predicted_TC_WT_ratio = predicted_TC_volume/predicted_WT_volume
    predicted_ET_WT_ratio = predicted_ET_volume/predicted_WT_volume
    predicted_ET_TC_ratio = predicted_ET_volume/predicted_TC_volume
    
    tumor_volume[_id] = [Edema, 
                         GD_enhancing_tumor_core, 
                         necrotic_tumor_core, 
                         WT_volume, 
                         TC_volume, 
                         ET_volume, 
                         TC_WT_ratio, 
                         ET_WT_ratio, 
                         ET_TC_ratio,
                         predicted_Edema, 
                         predicted_GD_enhancing_tumor_core, 
                         predicted_necrotic_tumor_core, 
                         predicted_WT_volume, 
                         predicted_TC_volume, 
                         predicted_ET_volume, 
                         predicted_TC_WT_ratio, 
                         predicted_ET_WT_ratio, 
                         predicted_ET_TC_ratio,]
tumor_volume_df = pd.DataFrame.from_dict(tumor_volume, 
                                         orient = 'index', 
                                         columns = ['ED', 
                                                     'ET', 
                                                     'NCR', 
                                                     'WT_volume', 
                                                     'TC_volume', 
                                                     'ET_volume', 
                                                     'TC_WT_ratio', 
                                                     'ET_WT_ratio', 
                                                     'ET_TC_ratio',
                                                     'predicted_ED', 
                                                     'predicted_ET', 
                                                     'predicted_NCR', 
                                                     'predicted_WT_volume', 
                                                     'predicted_TC_volume', 
                                                     'predicted_ET_volume', 
                                                     'predicted_TC_WT_ratio', 
                                                     'predicted_ET_WT_ratio', 
                                                     'predicted_ET_TC_ratio'])

tumor_volume_df.to_csv('../Results/Analysis_Results/volume/GLI-Tumor_volumns-New.csv')

In [ ]:
tumor_volume_df